# London and Bristol: exploratory data analysis

**Summary.** London has higher advertised nightly prices than Bristol. Missing prices and entirely empty fields require explicit handling. Quotation and revenue-derived fields can reveal the target and must be excluded from predictors.

**Uncertainty.** Missing-price causes, the validity of extreme prices and performance in an unseen location remain unresolved.

One row represents one listing snapshot. Prices are GBP per night. This is descriptive analysis; no price model is fitted and no third-city data is accessed.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import os
import platform
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

# Local paths and Databricks volume paths share the same manifest structure.
IN_DATABRICKS = 'DATABRICKS_RUNTIME_VERSION' in os.environ or 'dbutils' in globals()
if IN_DATABRICKS:
    dbutils.widgets.text('input_root', '/Volumes/workspace/airbnb_eda/raw')
    dbutils.widgets.text('output_root', '/Volumes/workspace/airbnb_eda/artifacts')
    INPUT_ROOT = Path(dbutils.widgets.get('input_root'))
    OUTPUT_ROOT = Path(dbutils.widgets.get('output_root'))
else:
    PROJECT_ROOT = Path.cwd()
    if not (PROJECT_ROOT / 'sources.json').exists():
        PROJECT_ROOT = PROJECT_ROOT.parent
    INPUT_ROOT = Path(os.environ.get('AIRBNB_INPUT_ROOT', str(PROJECT_ROOT)))
    OUTPUT_ROOT = Path(os.environ.get('AIRBNB_OUTPUT_ROOT', str(PROJECT_ROOT / 'docs')))
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
FIGURE_ROOT = OUTPUT_ROOT / 'figures'
FIGURE_ROOT.mkdir(exist_ok=True)
manifest = json.loads((INPUT_ROOT / 'sources.json').read_text())
sources = manifest['sources']
if len(sources) != 2 or {s['city'] for s in sources} != {'london', 'bristol'}:
    raise ValueError('EDA accepts exactly the London and Bristol source records.')
for s in sources:
    expected = f"https://data.insideairbnb.com/united-kingdom/england/{s['city']}/{s['snapshot_date']}/data/listings.csv.gz"
    if s['url'] != expected or s['relative_path'] != f"data/raw/{s['city']}/{s['snapshot_date']}/listings.csv.gz":
        raise ValueError('Unexpected source URL or input path.')

plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False})
pd.set_option('display.max_rows', 100)

def table(frame):
    """Render aggregate tables without optional formatting dependencies."""
    f = frame.copy().astype(object).where(frame.notna(), 'NA')
    def clean(v):
        return str(v).replace('|', '/').replace('\n', ' ')
    return '\n'.join(['| ' + ' | '.join(map(clean, f.columns)) + ' |',
                      '| ' + ' | '.join(['---'] * len(f.columns)) + ' |'] +
                     ['| ' + ' | '.join(map(clean, row)) + ' |' for row in f.itertuples(index=False, name=None)])

def show(frame):
    display(Markdown(table(frame)))


CITY_ORDER = ['london', 'bristol']
CITY_COLORS = {'london': '#315B78', 'bristol': '#C47B43'}
plt.rcParams.update({'font.size': 11, 'axes.titlesize': 13, 'axes.labelsize': 11,
                     'figure.facecolor': 'white', 'axes.facecolor': 'white'})
report_sections = []

def finding(text):
    display(Markdown(text))
    report_sections.append(text)

def figure_result(fig, filename, title, interpretation):
    fig.savefig(FIGURE_ROOT / filename, dpi=150, bbox_inches='tight')
    plt.show()
    display(Markdown(interpretation))
    report_sections.append(f'### {title}\n\n![{title}](figures/{filename})\n\n{interpretation}')

## 1. Dataset coverage

The inputs are detailed Inside Airbnb listing files. Source dates and checksums identify the exact snapshots. Personal descriptions, names and URLs are excluded from the analytical frame.

In [ ]:
CORE = ['id', 'price', 'room_type', 'accommodates', 'property_type', 'neighbourhood_cleansed']
NUMERIC = ['accommodates', 'bathrooms', 'bedrooms', 'beds', 'latitude', 'longitude',
           'minimum_nights', 'maximum_nights', 'availability_30', 'availability_60',
           'availability_90', 'availability_365', 'availability_eoy', 'number_of_reviews',
           'number_of_reviews_ltm', 'number_of_reviews_l30d', 'reviews_per_month',
           'review_scores_rating', 'review_scores_accuracy', 'review_scores_cleanliness',
           'review_scores_checkin', 'review_scores_communication', 'review_scores_location',
           'review_scores_value', 'estimated_occupancy_l365d']
CATEGORY = ['room_type', 'property_type', 'neighbourhood_cleansed', 'neighbourhood_group_cleansed',
            'host_is_superhost', 'instant_bookable', 'bathrooms_text', 'has_availability']
MONETARY = ['price', 'estimated_revenue_l365d', 'price_quote_total_price', 'price_quote_price_per_night']
used = ['id', 'last_scraped'] + NUMERIC + CATEGORY + MONETARY
used = list(dict.fromkeys(used))
frames, provenance, headers = [], [], {}
for s in sources:
    path = INPUT_ROOT / s['relative_path']
    with path.open('rb') as handle:
        digest = hashlib.file_digest(handle, 'sha256').hexdigest()
    if digest != s['sha256']:
        raise ValueError(f"Input checksum mismatch: {s['city']}")
    header = pd.read_csv(path, nrows=0).columns.tolist()
    headers[s['city']] = header
    absent = sorted(set(CORE) - set(header))
    if absent:
        raise ValueError(f"Required columns absent in {s['city']}: {absent}")
    frame = pd.read_csv(path, dtype='string', usecols=lambda c: c in used, keep_default_na=True)
    frame = frame.reindex(columns=used)
    frame['city'] = s['city']
    frame['snapshot_date'] = s['snapshot_date']
    frames.append(frame)
    provenance.append({'city': s['city'], 'snapshot_date': s['snapshot_date'], 'rows': len(frame),
                       'source_columns': len(header), 'loaded_columns': len(frame.columns) - 2,
                       'sha256_verified': True, 'source_url': s['url']})
raw = pd.concat(frames, ignore_index=True)
provenance = pd.DataFrame(provenance)
show(provenance[['city','snapshot_date','rows','source_columns']])

In [ ]:
coverage = provenance.set_index('city').loc[CITY_ORDER]
fig, ax = plt.subplots(figsize=(8, 3.4), layout='constrained')
ax.barh([c.title() for c in CITY_ORDER], coverage['rows'], color=[CITY_COLORS[c] for c in CITY_ORDER])
for i, value in enumerate(coverage['rows']):
    ax.text(value + 1000, i, f'{value:,}', va='center')
ax.set(xlabel='Listing snapshots', xlim=(0, coverage['rows'].max() * 1.2))
ax.invert_yaxis()
figure_result(fig, '01_dataset_coverage.png', 'London contributes most listing snapshots',
    f"London contains **{coverage.loc['london','rows']:,}** listings and Bristol **{coverage.loc['bristol','rows']:,}**. Combined summaries are therefore dominated by London; city comparisons are reported separately.")
report_sections.insert(0, '## Dataset and provenance\n\n' + table(provenance))

## 2. Missing data

A missing value means the source does not provide that attribute. Missingness is measured across all rows, before selecting records with an observed price.

In [ ]:
for c in raw.columns:
    raw[c] = raw[c].str.strip().replace('', pd.NA)
profile_rows = []
for city, frame in raw.groupby('city', sort=True):
    for c in used:
        present = c in headers[city]
        series = frame[c] if c in frame else pd.Series(pd.NA, index=frame.index, dtype='string')
        profile_rows.append({'city': city, 'column': c, 'present': present, 'source_type': 'CSV text',
                             'analysis_type': 'Float64' if c in NUMERIC or c in MONETARY else 'string',
                             'non_null': int(series.notna().sum()), 'missing_pct': round(100 * series.isna().mean(), 2)})
profile = pd.DataFrame(profile_rows)
grain_rows = []
for city, f in raw.groupby('city'):
    grain_rows.append({'city': city, 'rows': len(f), 'missing_id': int(f['id'].isna().sum()),
        'duplicate_key_rows': int(f.duplicated(['id', 'snapshot_date'], keep=False).sum()),
        'scrape_date_differs_from_snapshot_label': int((f['last_scraped'].notna() & f['last_scraped'].ne(f['snapshot_date'])).sum())})
grain = pd.DataFrame(grain_rows)
pattern_columns = [c for c in ['price','bathrooms','bedrooms','beds','review_scores_rating','reviews_per_month','neighbourhood_group_cleansed'] if c in raw]
patterns = []
for city, f in raw.groupby('city'):
    labels = f[pattern_columns].isna().apply(lambda r: ', '.join(r.index[r]) or 'complete', axis=1)
    counts = labels.value_counts().rename_axis('missing_columns').reset_index(name='rows')
    counts['city'] = city
    counts['pct'] = (100 * counts['rows'] / len(f)).round(2)
    patterns.append(counts)
patterns = pd.concat(patterns, ignore_index=True)

scrape_dates = raw.groupby(['city','last_scraped'],dropna=False).size().reset_index(name='rows')

In [ ]:
def parse_money(series):
    text = series.astype('string').str.strip()
    valid = text.str.fullmatch(r'(?:(?:£|\$|GBP)\s*)?[+-]?(?:\d+|\d{1,3}(?:,\d{3})+)(?:\.\d{1,2})?', na=False)
    numeric = text.where(valid).str.replace(r'^(?:£|\$|GBP)\s*', '', regex=True).str.replace(',', '', regex=False)
    return pd.to_numeric(numeric, errors='coerce').astype('Float64'), text.notna() & ~valid

analysis = raw.copy()
conversion_rows = []
for c in NUMERIC:
    if c in raw:
        analysis[c] = pd.to_numeric(raw[c], errors='coerce').astype('Float64')
        for city, idx in raw.groupby('city').groups.items():
            conversion_rows.append({'city': city, 'column': c, 'parse_failures': int((raw.loc[idx,c].notna() & analysis.loc[idx,c].isna()).sum())})
for c in MONETARY:
    if c in raw:
        analysis[c + '_gbp'], analysis[c + '_parse_failure'] = parse_money(raw[c])
price = analysis['price_gbp']
flags = pd.DataFrame(index=analysis.index)
flags['missing_price'] = raw['price'].isna()
flags['malformed_price'] = analysis['price_parse_failure']
flags['zero_price'] = price.eq(0).fillna(False)
flags['negative_price'] = price.lt(0).fillna(False)
flags['price_below_10'] = (price.gt(0) & price.lt(10)).fillna(False)
flags['price_above_2000'] = price.gt(2000).fillna(False)
flags['missing_id'] = raw['id'].isna()
flags['duplicate_key'] = raw.duplicated(['city','id','snapshot_date'], keep=False)
flags['nonpositive_or_fractional_accommodates'] = (analysis['accommodates'].le(0) | analysis['accommodates'].mod(1).ne(0)).fillna(False)
for c, limit in [('availability_30',30),('availability_60',60),('availability_90',90),('availability_365',365)]:
    if c in analysis:
        flags[c + '_invalid'] = (analysis[c].lt(0) | analysis[c].gt(limit) | analysis[c].mod(1).ne(0)).fillna(False)
flags['availability_nonmonotonic'] = False
for a,b in [('availability_30','availability_60'),('availability_60','availability_90'),('availability_90','availability_365')]:
    if a in analysis and b in analysis:
        flags['availability_nonmonotonic'] |= analysis[a].gt(analysis[b]).fillna(False)
for c in ['bathrooms','bedrooms','beds','number_of_reviews']:
    if c in analysis:
        flags[c + '_negative'] = analysis[c].lt(0).fillna(False)
if 'review_scores_rating' in analysis:
    flags['rating_outside_0_5'] = (analysis['review_scores_rating'].lt(0) | analysis['review_scores_rating'].gt(5)).fillna(False)
quality = flags.assign(city=analysis['city']).groupby('city').sum().reset_index()
conversions = pd.DataFrame(conversion_rows)
positive = analysis.loc[price.gt(0).fillna(False)].copy()
summary = positive.groupby('city')['price_gbp'].agg(['count','mean','median','min','max']).reset_index()
quantiles = positive.groupby('city')['price_gbp'].quantile([.25,.75,.95,.99]).unstack().reset_index()
quantiles.columns = ['city','p25','p75','p95','p99']
summary = summary.merge(quantiles,on='city')
summary['upper_iqr_fence'] = summary['p75'] + 1.5 * (summary['p75'] - summary['p25'])
fences = summary.set_index('city')['upper_iqr_fence']
analysis['price_upper_iqr_flag'] = price.gt(analysis['city'].map(fences)).fillna(False)
summary['above_iqr_rows'] = summary['city'].map(analysis.groupby('city')['price_upper_iqr_flag'].sum())
review_pattern = analysis.assign(has_reviews=analysis['number_of_reviews'].gt(0).fillna(False),
                                 missing_rating=analysis['review_scores_rating'].isna()).groupby(['city','has_reviews']).agg(
                                     rows=('id','size'), missing_rating_pct=('missing_rating', lambda s: 100*s.mean())).reset_index()

price_missingness = analysis.assign(
    missing_price=analysis['price_gbp'].isna(),
    availability_group=analysis['has_availability'].fillna('missing'),
    zero_available_days=analysis['availability_365'].eq(0).fillna(False)
).groupby(['city','availability_group','zero_available_days'],dropna=False).agg(
    rows=('id','size'),missing_price_pct=('missing_price',lambda x:100*x.mean())).reset_index()
bathroom_text = raw['bathrooms_text']
bathroom_count = pd.to_numeric(bathroom_text.str.extract(r'^(\d+(?:\.\d+)?)\s+(?:shared\s+|private\s+)?baths?$', expand=False),errors='coerce').astype('Float64')
half_bath = bathroom_text.str.fullmatch(r'(?i)(?:shared |private )?half-bath',na=False)
bathroom_count = bathroom_count.mask(half_bath,0.5)
bathroom_audit = pd.DataFrame({'city':raw['city'],'numeric_missing':analysis['bathrooms'].isna(),
    'text_present':bathroom_text.notna(),'recoverable_from_text':bathroom_count.notna(),
    'unrecognised_text':bathroom_text.notna() & bathroom_count.isna()}).groupby('city').sum().reset_index()

In [ ]:
fields = ['price', 'bathrooms', 'bedrooms', 'beds', 'review_scores_rating', 'instant_bookable', 'neighbourhood_group_cleansed']
labels = ['Nightly price', 'Numeric bathrooms', 'Bedrooms', 'Beds', 'Review rating', 'Instant booking', 'Neighbourhood group']
matrix = profile[profile['column'].isin(fields)].pivot(index='column', columns='city', values='missing_pct').loc[fields,CITY_ORDER]
fig, ax = plt.subplots(figsize=(8, 4.5), layout='constrained')
heat = ax.imshow(matrix.to_numpy(dtype=float), cmap='Blues', vmin=0, vmax=100, aspect='auto')
ax.set_xticks([0,1], ['London','Bristol'])
ax.set_yticks(range(len(labels)), labels)
for row in range(len(fields)):
    for col in range(2):
        value = float(matrix.iloc[row,col])
        ax.text(col,row,f'{value:.1f}%',ha='center',va='center',color='white' if value>55 else '#172A3A')
fig.colorbar(heat,ax=ax,label='Missing rows (%)',shrink=.8)
figure_result(fig, '02_missing_fields.png', 'Missingness differs substantially by field',
    f"Price is missing for **{matrix.loc['price','london']:.2f}%** of London rows and **{matrix.loc['price','bristol']:.2f}%** of Bristol rows. Numeric bathrooms, instant booking and neighbourhood-group labels are entirely missing in both snapshots. Bathroom text permits recovery of **{int(bathroom_audit.set_index('city').loc['london','recoverable_from_text']):,}** London and **{int(bathroom_audit.set_index('city').loc['bristol','recoverable_from_text']):,}** Bristol counts.")

In [ ]:
subset = price_missingness[price_missingness['availability_group'].eq('t')]
fig, ax = plt.subplots(figsize=(8, 3.8), layout='constrained')
x = np.arange(2)
for offset, zero in [(-.18,False),(.18,True)]:
    f = subset[subset['zero_available_days'].eq(zero)].set_index('city').loc[CITY_ORDER]
    bars=ax.bar(x+offset,f['missing_price_pct'],width=.34,label='Zero available days' if zero else 'At least one available day',color='#8B9FAD' if zero else '#315B78')
    for bar, n in zip(bars,f['rows']):
        ax.text(bar.get_x()+bar.get_width()/2,bar.get_height()+2,f'n={n:,}',ha='center',fontsize=9)
ax.set(xticks=x,xticklabels=['London','Bristol'],ylabel='Missing prices (%)',ylim=(0,118))
ax.legend(loc='lower center',bbox_to_anchor=(.5,1.02),ncol=2,fontsize=9,frameon=False)
figure_result(fig, '03_price_missingness.png', 'Missing prices are associated with calendar availability',
    'Among listings marked as having availability, nearly all rows with zero available days lack a price. Missing prices are much less common when at least one day is available. This association does not establish the cause; observed-price summaries may describe a selected subset of listings.')

## 3. Nightly advertised prices

Currency symbols and thousands separators are validated before conversion. Missing targets remain missing. Price comparisons use positive parsed prices; all raw rows remain preserved.

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(11,4),layout='constrained')
maximum=float(positive['price_gbp'].max())
minimum=float(positive['price_gbp'].min())
bins=np.geomspace(minimum,maximum,46)
for city in CITY_ORDER:
    values=positive.loc[positive['city'].eq(city),'price_gbp'].astype(float)
    axes[0].hist(values,bins=bins,weights=np.ones(len(values))*100/len(values),alpha=.55,color=CITY_COLORS[city],label=f'{city.title()} (n={len(values):,})')
axes[0].set(xscale='log',xlabel='Nightly price (GBP; logarithmic scale)',ylabel='Priced listings per bin (%)')
axes[0].set_xticks([10,50,100,500,2000,10000,75000], ['10','50','100','500','2k','10k','75k'])
axes[0].legend(fontsize=9)
medians=summary.set_index('city').loc[CITY_ORDER,'median'].astype(float)
axes[1].bar(['London','Bristol'],medians,color=[CITY_COLORS[c] for c in CITY_ORDER])
for i,value in enumerate(medians): axes[1].text(i,value+4,f'£{value:.0f}',ha='center')
axes[1].set(ylabel='Median nightly price (GBP)',ylim=(0,float(medians.max())*1.22))
figure_result(fig, '04_price_distribution.png', 'London has a higher median advertised price',
    f"The median is **£{medians.loc['london']:.0f}** in London and **£{medians.loc['bristol']:.0f}** in Bristol. The logarithmic axis preserves the full positive price range. Higher means than medians indicate a long upper tail; the city difference is descriptive rather than an adjusted city effect.")
small=summary[['city','count','mean','median','min','max']].round(2)
show(small)
report_sections.append(table(small))

In [ ]:
review_flags = quality.set_index('city').loc[CITY_ORDER,'price_above_2000']
finding(f"**Extreme-value review.** Prices above £2,000 occur in **{int(review_flags.loc['london']):,}** London and **{int(review_flags.loc['bristol']):,}** Bristol records. The maximum is **£{maximum:,.0f}**. These are review flags, not confirmed errors; no automatic trimming or capping is applied. No observed price is malformed, zero or negative in the pinned snapshots.")

## 4. Listing characteristics

Room type, guest capacity and neighbourhood provide context for the price difference. Each segment uses its own observed-price denominator.

In [ ]:
segment_tables = {}
for segment in ['room_type','accommodates','neighbourhood_cleansed']:
    grouped = positive.groupby(['city',segment], dropna=False)['price_gbp'].agg(['size','median','mean']).reset_index()
    grouped.columns = ['city',segment,'priced_rows','median_gbp','mean_gbp']
    if segment == 'neighbourhood_cleansed':
        grouped = grouped.sort_values(['city','priced_rows'],ascending=[True,False]).groupby('city').head(10)
    segment_tables[segment] = grouped
room_mix = analysis.groupby(['city','room_type'],dropna=False).size().reset_index(name='rows')
room_mix['pct'] = (100 * room_mix['rows'] / room_mix.groupby('city')['rows'].transform('sum')).round(2)
availability = analysis.groupby('city')['availability_365'].agg(['count','mean','median','min','max']).reset_index()
review_summary = analysis.groupby('city')['review_scores_rating'].agg(['count','mean','median']).reset_index()

In [ ]:
fig, axes=plt.subplots(1,2,figsize=(11,4.3),layout='constrained')
room_order=['Entire home/apt','Private room','Hotel room','Shared room']
max_median=float(segment_tables['room_type']['median_gbp'].max())
for ax,city in zip(axes,CITY_ORDER):
    f=segment_tables['room_type'].query('city == @city').set_index('room_type').reindex(room_order)
    ax.barh(room_order,f['median_gbp'].astype(float),color=CITY_COLORS[city])
    for i,row in enumerate(f.itertuples()):
        if pd.notna(row.median_gbp): ax.text(float(row.median_gbp)+3,i,f'n={int(row.priced_rows):,}',va='center',fontsize=9)
    ax.set(title=city.title(),xlabel='Median nightly price (GBP)',xlim=(0,max_median*1.3))
    ax.invert_yaxis()
figure_result(fig, '05_room_type.png', 'Room type separates different price segments',
    'Entire homes have higher median prices than private rooms in both cities. Compare the same room type across cities before interpreting an overall price difference. Hotel and shared-room estimates have small denominators, especially in Bristol.')

In [ ]:
fig,ax=plt.subplots(figsize=(8,3.8),layout='constrained')
for city in CITY_ORDER:
    f=segment_tables['accommodates'].query('city == @city').sort_values('accommodates')
    f=f[f['accommodates'].between(1,8)]
    ax.plot(f['accommodates'].astype(float),f['median_gbp'].astype(float),marker='o',color=CITY_COLORS[city],label=city.title())
ax.set(xlabel='Maximum guests',ylabel='Median nightly price (GBP)',xticks=range(1,9))
ax.legend()
figure_result(fig,'06_guest_capacity.png','Larger guest capacity is associated with higher median prices',
    'Median prices generally increase with guest capacity. The chart shows capacities of one to eight guests for readability; all capacities and segment counts remain in the technical audit. This association does not imply a causal effect.')

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(12,5),layout='constrained')
for ax,city in zip(axes,CITY_ORDER):
    f=segment_tables['neighbourhood_cleansed'].query('city == @city').head(8).sort_values('median_gbp')
    ax.barh(f['neighbourhood_cleansed'],f['median_gbp'].astype(float),color=CITY_COLORS[city])
    ax.set(title=city.title(),xlabel='Median nightly price (GBP)',xlim=(0,350))
figure_result(fig,'07_neighbourhoods.png','Prices vary within each city',
    'Neighbourhood medians vary substantially within each city. The chart includes eight neighbourhoods with the most observed prices per city, not the eight most expensive. Differences also reflect property and room-type composition.')

## 5. Availability and review scores

Availability counts days offered in the calendar. Review summaries include only non-missing ratings; listings without reviews are not assigned a score.

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(11,4),layout='constrained')
for city in CITY_ORDER:
    f=analysis[analysis['city'].eq(city)]
    available=f['availability_365'].dropna().astype(float)
    ratings=f['review_scores_rating'].dropna().astype(float)
    axes[0].hist(available,bins=np.linspace(0,365,25),weights=np.ones(len(available))*100/len(available),alpha=.55,color=CITY_COLORS[city],label=city.title())
    axes[1].hist(ratings,bins=np.linspace(0,5,26),weights=np.ones(len(ratings))*100/len(ratings),alpha=.55,color=CITY_COLORS[city],label=f'{city.title()} (n={len(ratings):,})')
axes[0].set(xlabel='Available days in the next 365 days',ylabel='Listings per bin (%)')
axes[1].set(xlabel='Review rating (0–5)',ylabel='Rated listings per bin (%)',xlim=(0,5))
for ax in axes: ax.legend(fontsize=9)
av=availability.set_index('city')
rs=review_summary.set_index('city')
figure_result(fig,'08_availability_reviews.png','Availability differs while observed review ratings are generally high',
    f"Median calendar availability is **{av.loc['london','median']:.0f} days** in London and **{av.loc['bristol','median']:.0f} days** in Bristol. Mean observed ratings are **{rs.loc['london','mean']:.2f}** and **{rs.loc['bristol','mean']:.2f}**. Missing overall ratings occur among listings with no recorded reviews; calendar availability must not be interpreted as occupancy.")

## 6. Price leakage

A predictor must not contain the answer it is intended to estimate. Quotation fields and price-derived revenue estimates are audited separately from candidate listing attributes.

In [ ]:
leakage_rows = []
for city, header in headers.items():
    for c in header:
        if any(t in c.lower() for t in ['price','revenue','occupancy']) or c == 'review_scores_value':
            reason = ('Target' if c == 'price' else
                      'Price quotation or quotation metadata' if c.startswith('price_quote_') else
                      'Monetary estimate may directly encode target price' if 'revenue' in c else
                      'Estimation assumptions and demand endogeneity' if 'occupancy' in c else
                      'Value judgement depends on advertised price')
            leakage_rows.append({'city':city,'column':c,'decision':'exclude','reason':reason})
leakage = pd.DataFrame(leakage_rows)
monetary_audit = []
for city,f in analysis.groupby('city'):
    for c in MONETARY:
        if c == 'price' or c + '_gbp' not in f: continue
        paired = f[['price_gbp', c+'_gbp']].dropna().astype(float)
        monetary_audit.append({'city':city,'column':c,'paired_rows':len(paired),
            'pearson_with_price': paired.corr().iloc[0,1] if len(paired)>1 else np.nan,
            'equal_to_price_rows':int(paired.iloc[:,0].eq(paired.iloc[:,1]).sum())})
monetary_audit = pd.DataFrame(monetary_audit)
rule_rows = []
for city,f in analysis.groupby('city'):
    idx=f.index
    for flag in flags:
        n=int(flags.loc[idx,flag].sum())
        rule_rows.append({'city':city,'rule':flag,'affected_rows':n,'pct':round(100*n/len(f),2)})
    rule_rows.append({'city':city,'rule':'upper_iqr_review_only','affected_rows':int(f['price_upper_iqr_flag'].sum()),'pct':round(100*f['price_upper_iqr_flag'].mean(),2)})
rule_impacts = pd.DataFrame(rule_rows)

revenue_proxy_audit = []
for city,f in analysis.groupby('city'):
    occupancy = f['estimated_occupancy_l365d']
    ratio = f['estimated_revenue_l365d_gbp'].div(occupancy.where(occupancy.gt(0)))
    pair = pd.DataFrame({'price':f['price_gbp'],'revenue_per_occupied_day':ratio}).dropna().astype(float)
    revenue_proxy_audit.append({'city':city,'paired_rows':len(pair),
        'pearson_with_price':pair.corr().iloc[0,1] if len(pair)>1 else np.nan,
        'equal_within_1p_rows':int(np.isclose(pair.iloc[:,0],pair.iloc[:,1],atol=.01,rtol=0).sum())})
revenue_proxy_audit = pd.DataFrame(revenue_proxy_audit)

In [ ]:
quote=monetary_audit[monetary_audit['column'].eq('price_quote_price_per_night')].set_index('city').loc[CITY_ORDER]
proxy=revenue_proxy_audit.set_index('city').loc[CITY_ORDER]
fig,ax=plt.subplots(figsize=(9,4),layout='constrained')
x=np.arange(2)
for offset,values,label,color in [(-.18,100*quote['equal_to_price_rows']/quote['paired_rows'],'Quotation equals price','#9F4545'),(.18,100*proxy['equal_within_1p_rows']/proxy['paired_rows'],'Revenue / occupancy within £0.01 of price','#B9A39A')]:
    bars=ax.bar(x+offset,values,width=.34,label=label,color=color)
    for bar,value in zip(bars,values): ax.text(bar.get_x()+bar.get_width()/2,value+2,f'{value:.1f}%',ha='center',fontsize=10)
ax.set(xticks=x,xticklabels=['London','Bristol'],ylabel='Matching paired rows (%)',ylim=(0,128))
ax.legend(loc='upper center',bbox_to_anchor=(.5,1.04),fontsize=9)
figure_result(fig,'09_price_leakage.png','Quotation and revenue-derived fields reveal the target',
    'The per-night quotation equals the target on every paired row. Revenue divided by estimated occupied days also closely reconstructs price. Exclude every `price_quote_*` field, revenue estimates and derived ratios; strong apparent predictive performance from these fields would be misleading.')
counts=pd.DataFrame({'city':CITY_ORDER,'quotation_pairs':quote['paired_rows'].to_numpy(),'revenue_occupancy_pairs':proxy['paired_rows'].to_numpy()})
show(counts)
report_sections.append(table(counts))

## 7. Cleaning and modelling implications

| Evidence | Recommended treatment |
| --- | --- |
| Missing target prices | Preserve records; exclude missing outcomes from supervised evaluation |
| Entirely missing attributes | Omit instant booking and neighbourhood-group labels from the initial model |
| Numeric bathrooms missing | Parse bathroom text, retaining unknown values and audit flags |
| Extreme positive prices | Retain and review; do not apply unvalidated caps |
| Price quotations and revenue proxies | Exclude from predictors |
| Missing review history | Preserve missing scores; do not replace them with low ratings |

Initial candidate predictors are room type, property type, guest capacity, bedrooms, beds, parsed bathrooms, latitude, longitude and minimum/maximum nights. City and neighbourhood require an unseen-category policy. Fit imputation, category vocabularies, thresholds and baseline medians on training rows only. Reserve the test set before model selection.

The detailed cleaning specification and audit tables document rule impacts. No production ETL or model is implemented by this notebook.

In [ ]:
findings = [
    '# London and Bristol: exploratory analysis',
    f"Execution: {datetime.now(timezone.utc).isoformat()}. Prices are nightly advertised GBP amounts. One row is one listing snapshot.",
    '## Summary',
    'London has higher observed advertised prices than Bristol. Missing targets and empty fields require explicit handling. Price quotations and revenue-derived fields must be excluded from predictors. Missing-price causes, extreme-price validity and out-of-city performance remain unresolved.',
] + report_sections + [
    '## Cleaning and modelling implications',
    'Preserve raw records and quality flags. Retain missing targets without replacing them with zero; exclude them only from supervised evaluation. Recover bathrooms from text. Omit entirely missing attributes from the initial model. Review extreme positive prices without automatic capping. Exclude quotation/revenue-derived predictors. Fit preprocessing and baseline statistics on training data only, after reserving the test set.',
    '## Limitations',
    'Published snapshot labels differ by five days and do not equal every listing scrape date. Comparisons are descriptive, price summaries use observed positive targets, small segments are unstable, and calendar availability is not occupancy. No model error or third-city transfer result has been measured.',
    '## Technical evidence',
    '[Full audit tables](eda_audit.md), [used-column profile](eda_profile.md), [cleaning specification](cleaning_specification.md). Source dates, URLs and SHA-256 checksums are recorded in sources.json. Inside Airbnb data are licensed CC BY 4.0.',
]
(OUTPUT_ROOT/'eda_findings.md').write_text('\n\n'.join(findings)+'\n')

## Technical appendix

The following cell exports the complete schema, missingness patterns, price-quality flags, segment statistics and leakage audit. These tables support reproducibility and do not change the visual findings above.

In [ ]:
report = [
    '# London and Bristol: technical EDA audit',
    f"Execution completed: {datetime.now(timezone.utc).isoformat()}. Python {platform.python_version()}; pandas {pd.__version__}; NumPy {np.__version__}.",
    '## Findings and limitations',
    f"London has {analysis['city'].eq('london').sum():,} rows and Bristol {analysis['city'].eq('bristol').sum():,}; median positive prices are £{summary.set_index('city').loc['london','median']:.2f} and £{summary.set_index('city').loc['bristol','median']:.2f}.",
    'The tables below describe all A/B snapshot rows and positive parsed prices. No raw rows were removed or overwritten. Prices are advertised nightly amounts in GBP, not realised booking revenue. City snapshots differ by five days. Model error and transfer performance remain unmeasured.',
    '## Dataset provenance and grain', table(provenance), table(grain), table(scrape_dates), 'Published snapshot labels and actual listing scrape dates differ. Preserve both; do not treat the difference alone as a corrupt record.',
    '## Price distribution (GBP)', table(summary.round(2)),
    'Positive parsed targets form the denominator for price statistics. Quality flags below cover all source rows.',
    '## Quality flags and cleaning impact', table(rule_impacts),
    'Flag counts overlap and must not be summed into a deletion count. Values below £10 or above £2,000 and upper-IQR flags require review rather than automatic exclusion.',
    '## Column profile', 'The complete used-column schema and missingness audit is in [eda_profile.md](eda_profile.md).',
    '## Missingness patterns', table(patterns.groupby('city').head(5)),
    'Pattern percentages use all rows in each city. Complete rows are included; combinations identify jointly absent fields.',
    '## Review-history missingness', table(review_pattern.round(2)), 'Missing ratings among rows without reviews are structurally plausible; unexplained gaps among reviewed listings require checking. Do not fabricate ratings.', '## Price missingness by availability',table(price_missingness.round(2)), 'This association does not identify the cause of missing prices. Do not infer occupancy from unavailable days or impute target prices from quote fields.', '## Bathroom recovery audit',table(bathroom_audit),
    '## Numeric parse failures', table(conversions[conversions.parse_failures > 0]),
    '## City comparisons', '### Calendar availability in the next 365 days (days)', table(availability.round(2)), '### Review rating (0–5)', table(review_summary.round(2)), '### Room-type composition', table(room_mix),
    'Availability is calendar availability, not occupancy. Review aggregates exclude missing scores and report their non-null denominator.',
    '## Segment prices (GBP)',
]
for segment,f in segment_tables.items():
    report.extend([f'### {segment}',table(f.round(2))])
report.extend([
    '## Leakage audit', table(leakage.drop(columns='city').drop_duplicates()), table(monetary_audit.round(4)), table(revenue_proxy_audit.round(4)),
    'Exclude all price quotations, quotation metadata, revenue estimates, perceived-value scores and price-derived features. Correlation is diagnostic and is not evidence that an unflagged field is safe. Revenue/occupancy ratios are prohibited features.',
    '## Verified leakage findings', 'The per-night quotation equals the target on every paired row in both cities. Revenue divided by estimated occupied days is also an almost exact price proxy. Neither belongs in the predictor set.',
    '## ETL recommendations',
    'Retain immutable Bronze files and provenance. In Silver, preserve rows with quality flags, quarantine invalid keys, parse GBP explicitly, retain missing values, and distinguish malformed from absent values. Exclude invalid targets only from supervised targets; keep their records for inference where possible. Review extremes without automatic capping. Validate capacity, availability and score domains. See cleaning_specification.md for rule rationales.',
    '## ML handoff',
    'Begin with room_type, property_type, accommodates, bedrooms, beds, bathrooms, latitude, longitude, minimum_nights and maximum_nights. Omit instant_bookable and neighbourhood_group_cleansed because both are entirely missing in the pinned snapshots; derive bathroom counts from text. Missing attributes need train-only imputation and indicators. City and neighbourhood require a pre-specified unseen-category policy. Reserve a test set per city before model selection; fit preprocessing, thresholds and baseline medians on training data only. No EDA summary is a test-set model metric.',
    '## Unresolved questions',
    'The publisher confirms that price uses local currency and that the dollar sign is an export artifact. Confirm newer quotation and revenue-estimation semantics against publisher metadata. Resolve structural missingness before selecting optional features. Production ETL, model evaluation and geographic transfer performance remain outside the scope of this exploratory analysis.',
    '## Source',
    '[Inside Airbnb data downloads](https://insideairbnb.com/get-the-data/), CC BY 4.0. Snapshot URLs and checksums are recorded in sources.json.',
])
report_text = '\n\n'.join(report) + '\n'
(OUTPUT_ROOT / 'eda_audit.md').write_text(report_text)
(OUTPUT_ROOT / 'eda_profile.md').write_text('# Used-column profile\n\n' + table(profile) + '\n\n## All missingness patterns\n\n' + table(patterns) + '\n')
(OUTPUT_ROOT / 'eda_run.json').write_text(json.dumps({
    'completed_utc': datetime.now(timezone.utc).isoformat(), 'python': platform.python_version(),
    'pandas': pd.__version__, 'numpy': np.__version__, 'sources':sources,
    'rows': {str(k):int(v) for k,v in analysis.groupby('city').size().items()},
    'raw_rows_removed':0, 'analysis_scope':['london','bristol']
},indent=2)+'\n')
print('Aggregate findings and technical audit exported.')